# Lab 2 · Classification: who will drop out?
**Goal:** at week 6, give each student a **probability** of dropping out, and decide whom teachers should call.

**You will practise**
- Handle a rare “yes” (11%)
- Stratified splits
- See why accuracy lies
- Find and remove a leak
- Compare models by ranking and by honest probabilities
- Check calibration
- Choose the cut-off from costs, not 0.5
- Use a capacity rule (“we can make 100 calls”)
- Test once with a range, read the missed cases
- Save the model **with** its cut-off

**How to work:** 🧠 = write your guess first · ✓ = the notebook proves the result

## 0 · Setup

In [ ]:
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, average_precision_score, brier_score_loss,
                             confusion_matrix, precision_recall_curve, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 0
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "font.size": 12})
print("pandas", pd.__version__, "· scikit-learn", sklearn.__version__)

## 1 · Load and look
- 3,000 students, all clues known at **week 6**
- Target: `dropped_out` (known at term end)
- Missing file? Run `python make_data.py`

In [ ]:
df = pd.read_csv("data/dropout.csv")
print(df.shape)
df.head()

In [ ]:
print("missing:", df.isna().sum()[df.isna().sum() > 0].to_dict())
print(f"dropout rate: {df['dropped_out'].mean():.1%}  ({df['dropped_out'].sum()} of {len(df)})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, col in zip(axes, ["attendance_wk6", "assignments_submitted", "logins_per_week"]):
    df.boxplot(column=col, by="dropped_out", ax=ax); ax.set_title(col); ax.set_xlabel("dropped out (0 = no, 1 = yes)")
plt.suptitle(""); plt.tight_layout(); plt.show()
print(df.groupby("part_time_job")["dropped_out"].mean().round(3).to_dict(), "← dropout rate by part-time job")

**What to notice**
- Dropouts attended less, handed in less, logged in less
- A part-time job more than doubles the risk
- Lots of overlap: no single clue decides

## 2 · 🧠 Predict first
- A lazy model says “nobody drops out”
- Its accuracy? How many dropouts does it catch?

In [ ]:
my_guess = {"accuracy": 0.89, "dropouts_caught": 0}

## 3 · Lock away a stratified test set
- A rare “yes” can end up unevenly split by chance
- `stratify=y` keeps the same dropout rate in both piles

In [ ]:
target = "dropped_out"
X = df.drop(columns=[target, "student_id"])
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
print(f"train {len(X_train)} (rate {y_train.mean():.1%}) · test {len(X_test)} (rate {y_test.mean():.1%}, {y_test.sum()} dropouts) – locked")

## 4 · Baseline: accuracy lies

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
lazy = DummyClassifier(strategy="most_frequent")
lazy_acc = cross_val_score(lazy, X_train, y_train, cv=cv).mean()
lazy_recall = cross_val_score(lazy, X_train, y_train, cv=cv, scoring="recall").mean()
print(f"'nobody drops out': accuracy {lazy_acc:.1%} · dropouts caught (recall) {lazy_recall:.0%}")
assert lazy_acc > 0.88 and lazy_recall == 0
print("✓ 89% accurate and completely useless: with a rare yes, never judge by accuracy")

**Which score, then?**
- **PR-AUC:** how well the rare “yes” is ranked first; lazy ≈ 0.11 (main score)
- **ROC-AUC:** chance a dropout ranks above a stayer; 0.5 = coin flip
- **Brier:** squared error of the probabilities; lower = more honest

## 5 · 🧠 Leak hunt: `exit_survey_done`
- How good does a model with only this column look?

In [ ]:
survey_only = Pipeline([("prep", ColumnTransformer([("s", OneHotEncoder(), ["exit_survey_done"])])),
                        ("model", LogisticRegression())])
print(f"exit survey only: PR-AUC {cross_val_score(survey_only, X_train, y_train, cv=cv, scoring='average_precision').mean():.2f}  (lazy ≈ {y_train.mean():.2f})")

**Too good to be true**
- The exit survey is filled in **after** a student leaves
- At week 6 it doesn't exist: a leak, drop it

In [ ]:
X_train = X_train.drop(columns="exit_survey_done"); X_test = X_test.drop(columns="exit_survey_done")
print("✓ leak removed:", list(X_train.columns))

## 6 · Pipeline
- Numbers: median for missing quiz scores **plus** a 0/1 “was missing” column (no quiz score is itself a warning), then standardise
- Categories: one-hot

In [ ]:
numeric = ["attendance_wk6", "assignments_submitted", "quiz_avg", "logins_per_week", "distance_km"]
categorical = ["stream", "part_time_job", "scholarship"]

def make_prep():
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), numeric),
        ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), categorical),
    ])

make_prep().fit(X_train).get_feature_names_out()

## 7 · 🧠 Compare models
- Logistic regression
- Logistic with `class_weight="balanced"` (pretends dropouts are as common as stayers)
- Random forest, boosting
- Which ranks best?

In [ ]:
my_best = "logistic"

In [ ]:
candidates = {
    "logistic": LogisticRegression(max_iter=1000),
    "logistic balanced": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "random forest": RandomForestClassifier(400, min_samples_leaf=5, random_state=SEED, n_jobs=-1),
    "boosting": HistGradientBoostingClassifier(random_state=SEED),
}
rows, cv_probs = {}, {}
for name, model in candidates.items():
    pipe = Pipeline([("prep", make_prep()), ("model", model)])
    p = cross_val_predict(pipe, X_train, y_train, cv=cv, method="predict_proba")[:, 1]   # out-of-fold probabilities
    cv_probs[name] = p
    rows[name] = {"PR-AUC": average_precision_score(y_train, p), "ROC-AUC": roc_auc_score(y_train, p),
                  "Brier": brier_score_loss(y_train, p), "recall @ 0.5": ((p >= 0.5) & (y_train == 1)).sum() / y_train.sum()}
table = pd.DataFrame(rows).T.sort_values("PR-AUC", ascending=False)
table.round(3)

In [ ]:
assert table.index[0] == "logistic" and table.loc["logistic", "PR-AUC"] > 2 * y_train.mean()
print("✓ plain logistic regression ranks best and its probabilities are the most honest (lowest Brier)")

**Read the table**
- All rank far better than chance, but at cut-off 0.5 most catch **under a third** of dropouts
- “Balanced”: same ranking, **inflated** probabilities (worse Brier)
- Better: keep honest probabilities and **choose the cut-off on purpose**

## 8 · Are the probabilities honest?
- Group students by predicted probability
- Did that share really drop out?

In [ ]:
p_cv = cv_probs["logistic"]
true_rate, predicted = calibration_curve(y_train, p_cv, n_bins=6, strategy="quantile")
plt.plot([0, 0.6], [0, 0.6], "--", c="grey", label="perfectly honest")
plt.plot(predicted, true_rate, "o-", label="logistic")
plt.xlabel("predicted probability"); plt.ylabel("real dropout rate"); plt.legend(); plt.show()
print(pd.DataFrame({"predicted": predicted, "real": true_rate}).round(2).to_string(index=False))

**Calibrated**
- Dots on the diagonal: “40%” really means about 40%
- Honest probabilities let us reason about costs

## 9 · 🧠 Choose the cut-off from costs
- A teacher's call costs **1**
- A missed dropout costs **10**
- Should the cut-off be above or below 0.5?

In [ ]:
my_cutoff_guess = 0.1

In [ ]:
COST_MISS, COST_CALL = 10, 1
cutoffs = np.round(np.arange(0.02, 0.81, 0.01), 2)
def total_cost(y_true, p, t):
    missed = ((p < t) & (y_true == 1)).sum()
    calls = (p >= t).sum()
    return COST_MISS * missed + COST_CALL * calls
costs = [total_cost(y_train.to_numpy(), p_cv, t) for t in cutoffs]
CUTOFF = float(cutoffs[int(np.argmin(costs))])
plt.plot(cutoffs, costs); plt.axvline(CUTOFF, c="green", ls="--", label=f"cheapest: {CUTOFF}"); plt.axvline(0.5, c="red", ls=":", label="default 0.5")
plt.xlabel("cut-off"); plt.ylabel("total cost (training, out-of-fold)"); plt.legend(); plt.show()
print(f"cheapest cut-off {CUTOFF} · cost {min(costs)} vs {total_cost(y_train.to_numpy(), p_cv, 0.5)} at 0.5")

In [ ]:
for t in (0.5, CUTOFF):
    tn, fp, fn, tp = confusion_matrix(y_train, p_cv >= t).ravel()
    print(f"cut-off {t:<4}: caught {tp}/{tp + fn} dropouts (recall {tp / (tp + fn):.0%}) · calls {tp + fp} · precision {tp / (tp + fp):.0%}")
assert CUTOFF < 0.2
print("✓ when missing a dropout costs 10× a call, the best cut-off is far below 0.5")

**Notice**
- The cost curve is flat near its minimum
- The exact cut-off matters less than **not using 0.5**

## 10 · Capacity rule: only 100 calls
- When time is the limit, call the 100 highest-risk students
- Ranking quality (PR-AUC) is what matters here

In [ ]:
top = np.argsort(-p_cv)[:100]
caught = y_train.to_numpy()[top].sum()
random_caught = 100 * y_train.mean()
print(f"top-100 by risk catch {caught} dropouts · 100 random students would catch ≈ {random_caught:.0f}")
assert caught > 4 * random_caught
print(f"✓ ranking by risk finds {caught / random_caught:.1f}× more dropouts than calling at random")

## 11 · Final test: once
- Train on all training students
- Score the locked test set with the chosen cut-off
- Give a range for recall

In [ ]:
final = Pipeline([("prep", make_prep()), ("model", LogisticRegression(max_iter=1000))]).fit(X_train, y_train)
p_test = final.predict_proba(X_test)[:, 1]
flag = p_test >= CUTOFF
tn, fp, fn, tp = confusion_matrix(y_test, flag).ravel()
recall, precision = tp / (tp + fn), tp / (tp + fp)

rng = np.random.default_rng(SEED); yt = y_test.to_numpy(); boot = []
for _ in range(3000):
    i = rng.integers(0, len(yt), len(yt)); boot.append((flag[i] & (yt[i] == 1)).sum() / max(1, (yt[i] == 1).sum()))
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"test: caught {tp} of {tp + fn} dropouts · recall {recall:.0%} (95% range {lo:.0%}–{hi:.0%}) · precision {precision:.0%} · calls {tp + fp}")
print(f"PR-AUC {average_precision_score(y_test, p_test):.2f} · ROC-AUC {roc_auc_score(y_test, p_test):.2f} · accuracy at 0.5 {((p_test >= 0.5) == y_test).mean():.1%} (lazy {1 - y_test.mean():.1%})")
ConfusionMatrixDisplay.from_predictions(y_test, flag, display_labels=["stays", "drops out"], cmap="Blues"); plt.title(f"Test set, cut-off {CUTOFF}"); plt.show()

In [ ]:
recall_at_half = ((p_test >= 0.5) & (yt == 1)).sum() / yt.sum()
print(f"same model, default cut-off 0.5: recall {recall_at_half:.0%}")
assert lo > recall_at_half
print(f"✓ proof: on students never seen we catch {recall:.0%} of dropouts (at least {lo:.0%} likely) vs {recall_at_half:.0%} at 0.5, calling {tp + fp} of {len(yt)} students")

## 12 · Read the missed dropouts
- What do they look like?

In [ ]:
missed = X_test[(~flag) & (y_test == 1)].assign(p=p_test[(~flag) & (y_test == 1)].round(3))
print(f"{len(missed)} missed dropouts; typical profile vs all dropouts:")
pd.DataFrame({"missed": missed[numeric].median(), "all dropouts": X_test[y_test == 1][numeric].median()}).round(1)

**What the misses say**
- They look like ordinary students at week 6
- The reason they left isn't in the data (family, money, health)
- To catch them: **new clues**, not a fancier model

## 13 · Explain: which clues raise the risk?
- Each weight = change in log-odds per standard deviation (numbers) or for that category

In [ ]:
weights = pd.Series(final.named_steps["model"].coef_[0], index=final.named_steps["prep"].get_feature_names_out()).sort_values()
weights.plot.barh(color=np.where(weights > 0, "#C2412D", "#2E7D5B")); plt.xlabel("← lowers risk · raises risk →"); plt.show()
assert weights.idxmin().endswith("attendance_wk6") and weights.idxmax().endswith("part_time_job_yes")
print("✓ low attendance and a part-time job raise the risk most; a scholarship lowers it")

## 14 · Save the model **with its cut-off**

In [ ]:
Path("models").mkdir(exist_ok=True)
joblib.dump({"pipeline": final, "cutoff": CUTOFF, "cost_ratio": f"{COST_MISS}:{COST_CALL}"}, "models/dropout_model.joblib")

saved = joblib.load("models/dropout_model.joblib")
def assess(student: dict) -> dict:
    p = float(saved["pipeline"].predict_proba(pd.DataFrame([student]))[0, 1])
    return {"risk": round(p, 3), "call": p >= saved["cutoff"]}

ravi = {"stream": "arts", "attendance_wk6": 62, "assignments_submitted": 55, "quiz_avg": 48,
        "logins_per_week": 2, "distance_km": 18, "part_time_job": "yes", "scholarship": "no"}
neha = {"stream": "science", "attendance_wk6": 94, "assignments_submitted": 95, "quiz_avg": 78,
        "logins_per_week": 7, "distance_km": 4, "part_time_job": "no", "scholarship": "yes"}
print("Ravi:", assess(ravi), "· Neha:", assess(neha))
assert assess(ravi)["call"] and not assess(neha)["call"]
print("✓ the saved model flags Ravi for a call and leaves Neha alone")

## 15 · Your turn
- Cost ratio 5:1 and 20:1: how do cut-off, recall and calls change?
- 150 calls: how many test dropouts does the top-150 catch?
- Add `job_and_low_attendance` (job AND attendance < 70): does CV PR-AUC improve?

In [ ]:
# TODO

## ✅ Checklist for every classification project
- [x] Measured how rare the “yes” is; stratified every split
- [x] Started from a lazy baseline; didn't judge by accuracy
- [x] Hunted leaks: *does this column exist at prediction time?*
- [x] Compared models by ranking (PR-AUC) and honesty (Brier), out-of-fold
- [x] Checked calibration
- [x] Chose the cut-off from costs (or capacity), on training data only
- [x] Tested once, with a range
- [x] Read the missed cases
- [x] Explained the weights
- [x] Saved the pipeline with its cut-off